# LinkedIn - LinkedIn Messaging: User Engagement Insights

In [1]:
import pandas as pd 
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_messages = pd.read_csv('../Data/012/fct_messages.csv', parse_dates=['message_sent_date'])

pl_messages = pl.read_csv('../Data/012/fct_messages.csv', try_parse_dates=True)

# Preguntas 1

### ¿Cuál es el número total de mensajes enviados durante abril de 2024? Esta información nos ayudará a cuantificar el compromiso (engagement) general como una línea base para mejoras focalizadas en el producto.

```SQL
SELECT
    COUNT(*) AS total_messages
FROM fct_messages
WHERE ((EXTRACT(MONTH FROM message_sent_date) = 4) AND
       (EXTRACT(YEAR FROM message_sent_date) = 2024));
```

In [4]:
abril = df_messages[
    (df_messages['message_sent_date'].dt.month == 4) &
    (df_messages['message_sent_date'].dt.year == 2024)
].shape[0]

abril

60

In [5]:
res = pl_messages.filter(
    (pl.col('message_sent_date').dt.month() == 4) &
    (pl.col('message_sent_date').dt.year() == 2024)
).height

res

60

# Pregunta 2

### ¿Cuál es el promedio de mensajes enviados por usuario durante abril de 2024? Redondea tu resultado al número entero más cercano. Esta métrica proporciona información sobre los niveles de compromiso individual para perfeccionar nuestras funciones de comunicación.

```SQL
SELECT
    ROUND(AVG(conteo_por_usuario), 0) AS promedio_mensajes
FROM (SELECT user_id,
             count(message_id) AS conteo_por_usuario
      FROM fct_messages
      WHERE ((EXTRACT(MONTH FROM message_sent_date) = 4) AND
             (EXTRACT(YEAR FROM message_sent_date) = 2024))
      GROUP BY user_id) AS subquery
```

In [9]:
abril = df_messages[
    (df_messages['message_sent_date'].dt.month == 4) &
    (df_messages['message_sent_date'].dt.year == 2024)
].groupby('user_id').agg(
    conteo_por_usuario = ('message_id', 'count')
).reset_index()

res = abril[['conteo_por_usuario']].agg(
    promedio_mensajes = ('conteo_por_usuario', 'mean')
).round(2)

res

,conteo_por_usuario
promedio_mensajes,20.0


In [11]:
res = pl_messages.filter(
    (pl.col('message_sent_date').dt.month() == 4) &
    (pl.col('message_sent_date').dt.year() == 2024)
).group_by('user_id').agg(
    pl.len().alias('conteo_por_usuario')
)

res_avg = res.select(
    pl.col('conteo_por_usuario').mean().round(2).alias('promedio_mensjaes')
)

res_avg

promedio_mensjaes
f64
20.0


# Pregunta 3

### ¿Qué porcentaje de usuarios envió más de 50 mensajes durante abril de 2024? Este cálculo ayudará a identificar a los usuarios altamente comprometidos y servirá de base para recomendaciones orientadas a mejorar las interacciones de mensajería.

```SQL
WITH conteo_usuarios AS (
    SELECT user_id,
            COUNT(message_id) AS total_mensajes
     FROM fct_messages
     WHERE ((EXTRACT(MONTH FROM message_sent_date) = 4) AND
            (EXTRACT(YEAR FROM message_sent_date) = 2024))
     GROUP BY user_id
)
SELECT
    ROUND(
        (COUNT(CASE WHEN total_mensajes > 50 THEN 1 END) * 100.0)/COUNT(*)
        ,2) AS pct_over_50
FROM conteo_usuarios;
```

In [24]:
abril = df_messages[
    (df_messages['message_sent_date'].dt.month == 4) &
    (df_messages['message_sent_date'].dt.year == 2024)
].reset_index()

res = abril.groupby('user_id').agg(
    total_mensajes = ('message_id', 'count')
).reset_index()

over_50 = (res['total_mensajes'] > 50).sum()

pct_over50 = round(((over_50 / res.shape[0]) * 100),2)

pct_over50

np.float64(33.33)

In [34]:
res = pl_messages.filter(
    (pl.col('message_sent_date').dt.month() == 4) &
    (pl.col('message_sent_date').dt.year() == 2024)
).group_by('user_id').agg(
    pl.len().alias('total_mensajes')
)

pct_res = res.select(
    ((pl.col('total_mensajes') > 50).sum() / pl.len() * 100)
    .round(2)
    .alias('pct_over50')
)

pct_res



pct_over50
f64
33.33
